<a href="https://colab.research.google.com/github/MichalSlowakiewicz/Reinforcement-Learning/blob/master/LAB_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a target="_blank" href="https://colab.research.google.com/github/mim-ml-teaching/public-rl-2025-26/blob/main/labs/RL_LAB6_policy_gradients_jax.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Mini-Tutorial: JAX Fundamentals

Before we dive into the policy gradients, we need to understand the tool we are using. **JAX** is a library for high-performance numerical computing. It is essentially "NumPy on steroids" (running on GPUs/TPUs) combined with composable function transformations like automatic differentiation and Just-In-Time (JIT) compilation.

However, JAX requires us to write code in a functional style. Here are the three most important differences you need to know for this homework:

### 1. Immutable Arrays and `jnp`
JAX provides a NumPy-like API via `jax.numpy` (usually imported as `jnp`). They act almost exactly like standard NumPy arrays, but they are **immutable**. You cannot change a value in-place (e.g., `x[0] = 5` will throw an error).

In [1]:
import numpy as np
import jax.numpy as jnp

x_np = np.array([1.0, 2.0, 3.0])
x_jnp = jnp.array([1.0, 2.0, 3.0])

# This is allowed in NumPy:
x_np[0] = 99.0

# In JAX, arrays are immutable! To update an array, you use the `.at[].set()` syntax,
# which returns a NEW array.
x_jnp_new = x_jnp.at[0].set(99.0)

print("Original JAX array:", x_jnp)
print("Updated JAX array: ", x_jnp_new)

Original JAX array: [1. 2. 3.]
Updated JAX array:  [99.  2.  3.]


### 2. Explicit Randomness (PRNG Keys)
In standard NumPy or PyTorch, random number generation is stateful and hidden globally. You just call `np.random.rand()` and it magically gives you a new number.

JAX does not do this because hidden state makes it impossible to parallelize and compile code safely. Instead, JAX requires an **explicit random state (a key)** to be passed to every random function. Whenever you need a new random number, you must explicitly **split** your key into a new subkey for the operation, and a new key to keep for the future.

In [2]:
import jax

key = jax.random.PRNGKey(42)
key, subkey = jax.random.split(key)

random_normal = jax.random.normal(subkey)
print("Random Number 1:", random_normal)
print("Reusing subkey: ", jax.random.normal(subkey))

key, subkey2 = jax.random.split(key)
print("Random Number 2:", jax.random.normal(subkey2))

Random Number 1: 0.60576403
Reusing subkey:  0.60576403
Random Number 2: -0.21089035


### 3. JIT Compilation (`jax.jit`)
JAX can compile your Python functions into highly optimized, fast C++/XLA code using the `@jax.jit` decorator.

However, **JIT-compiled functions must be pure**. They cannot have side effects (like appending to a global python list or printing inside the compiled loop), and they must return all their outputs explicitly.

In [3]:
import time

def slow_activation(x):
    return jnp.maximum(0.0, x) + jnp.exp(-x)

fast_activation = jax.jit(slow_activation)
data = jax.random.normal(key, (5000, 5000))

# First run of JIT takes longer because it has to compile the function
start = time.time()
_ = fast_activation(data)
print(f"First run (includes compilation time): {time.time() - start:.4f} seconds")

# Second run is lightning fast!
start = time.time()
_ = fast_activation(data)
print(f"Second run (compiled XLA execution): {time.time() - start:.4f} seconds")

First run (includes compilation time): 0.2022 seconds
Second run (compiled XLA execution): 0.0027 seconds


### 4. Automatic Differentiation (`jax.grad` and `jax.value_and_grad`)
In Reinforcement Learning, we need to take the gradient of our objective function with respect to our neural network parameters. JAX makes this incredibly easy.

`jax.grad(f)` returns a *new function* that computes the gradient of `f`. `jax.value_and_grad(f)` returns a function that gives you *both* the original output and the gradient, which is very useful for tracking loss during training.

In [4]:
def f(x):
    return x ** 3.0

# The analytical derivative is f'(x) = 3x^2
# Let's see what JAX calculates at x = 2.0. (Expected: 3 * (2^2) = 12.0)

f_and_grad = jax.value_and_grad(f)
value, gradient = f_and_grad(2.0)

print(f"Function value at x=2.0: {value}")
print(f"Gradient at x=2.0:       {gradient}")

Function value at x=2.0: 8.0
Gradient at x=2.0:       12.0


# Policy Gradients: REINFORCE and VPG

In this notebook, we will explore **Policy Gradients** from the ground up using JAX and Flax. We will start with the fundamental mathematical derivations and translate them directly into fast, compiled JAX code using a procedural style.

Policy Gradient algorithms optimize the policy directly by maximizing the expected return:

$$J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}[R(\tau)]$$

Where $\tau$ is a trajectory (a sequence of states and actions) and $R(\tau)$ is the total reward of that trajectory. The key idea of policy gradients is simple: **We want to increase the probability of actions that lead to high returns, and decrease the probability of actions that lead to low returns.**

Using the log-derivative trick ($\nabla x = x \nabla \log x$), we can derive the simplest form of the policy gradient:

$$\nabla_\theta J(\pi_\theta) = \mathbb{E}_{\tau \sim \pi_\theta} \left[ \sum_{t=0}^T \nabla_\theta \log \pi_\theta(a_t | s_t) R(\tau) \right]$$

Check out the derivation here https://spinningup.openai.com/en/latest/spinningup/rl_intro3.html

To implement this, we:
1. Run the agent in the environment to collect trajectories.
2. Calculate the log probability of the actions taken.
3. Multiply those by the returns.
4. Take the gradient and update the network weights.

Let's set up our JAX environment.

In [5]:
!pip install -q gymnasium flax optax

In [13]:
import os
import random
import numpy as np
import gymnasium as gym
import jax
import jax.numpy as jnp
import flax.linen as nn
import optax
import jax.lax as lx

# Set random seeds for reproducibility
seed = 42
key = jax.random.PRNGKey(seed)
np.random.seed(seed)
random.seed(seed)

## Task 1: Defining the Neural Network Policy

We need a neural network that takes an environment observation as input and outputs the probabilities of taking different actions.

**Your Task:**
Using `flax.linen`, define a simple Multi-Layer Perceptron (MLP) with two hidden layers of 64 units each and `tanh` activations. Because we are predicting probabilities for discrete actions, your network should output **logits** (unnormalized log-probabilities) of size `action_dim`.

In [10]:
class PolicyNetwork(nn.Module):
    action_dim: int

    @nn.compact
    def __call__(self, x):
        # ==========================================
        # YOUR CODE HERE
        # Define 2 hidden layers (64 units, tanh)
        # and a final dense layer for the logits
        # ==========================================
        x = nn.Dense(64)(x)
        x = nn.activation.tanh(x)
        x = nn.Dense(64)(x)
        x = nn.activation.tanh(x)
        logits = nn.Dense(self.action_dim)(x)
        return logits

## Task 2: The REINFORCE Algorithm (with Reward-to-Go)

In our original formula, we multiplied the log-probability of an action at time $t$ by the *total* trajectory return $R(\tau)$. However, an action taken at time $t$ cannot possibly affect the rewards received *before* time $t$.

We can reduce the variance of our gradients (making training much more stable) by replacing the total return with the **Reward-to-Go**:

$$\hat{R}_t = \sum_{t'=t}^T r_{t'}$$

**Your Task:**
Complete the `loss_fn` inside the `update_step`. You need to compute the log probabilities of the actions taken and multiply them by the rewards-to-go (`weight_batch`). Remember that policy gradients aim to *maximize* the objective, but standard optimizers *minimize* loss, so you will need to negate the mean.

In [11]:
def train_reinforce(env_id="CartPole-v1", epochs=10, batch_size=5000, lr=1e-2):
    env = gym.make(env_id)
    obs_dim = env.observation_space.shape[0]
    act_dim = env.action_space.n

    policy = PolicyNetwork(act_dim)
    global key
    key, subkey = jax.random.split(key)

    params = policy.init(subkey, jnp.zeros((1, obs_dim)))
    tx = optax.adam(learning_rate=lr)
    opt_state = tx.init(params)

    @jax.jit
    def update_step(params, opt_state, obs_batch, act_batch, weight_batch):
        def loss_fn(p):
            logits = policy.apply(p, obs_batch)

            # ==========================================
            # YOUR CODE HERE
            # 1. Get log probabilities using jax.nn.log_softmax
            # 2. Gather log_probs of the specific actions taken in the batch
            # 3. Calculate the policy gradient loss (don't forget the negative sign!)
            # ==========================================
            log_probs = jax.nn.log_softmax(logits)

            # gemini
            batch_size = logits.shape[0]
            selected_log_probs = log_probs[jnp.arange(batch_size), act_batch]
            loss = -jnp.mean(selected_log_probs * weight_batch)

            # laby
            #action_log_probs = log_probs[jnp.arange(len(act_batch)), act_batch]
            #loss = -jnp.mean(action_log_probs * weight_batch)


            return loss

        loss, grads = jax.value_and_grad(loss_fn)(params)
        updates, opt_state = tx.update(grads, opt_state, params)
        new_params = optax.apply_updates(params, updates)
        return new_params, opt_state, loss

    obs, _ = env.reset(seed=seed)

    for epoch in range(epochs):
        batch_obs, batch_acts, batch_weights, batch_rets, batch_lens = [], [], [], [], []
        ep_rews = []

        while True:
            batch_obs.append(obs)

            logits = policy.apply(params, jnp.array(obs))
            key, subkey = jax.random.split(key)
            act = jax.random.categorical(subkey, logits).item()

            obs, rew, terminated, truncated, _ = env.step(act)
            batch_acts.append(act)
            ep_rews.append(rew)

            if terminated or truncated:
                batch_rets.append(sum(ep_rews))
                batch_lens.append(len(ep_rews))

                # Compute Reward-to-Go
                rtgs = []
                discounted_sum = 0
                for r in reversed(ep_rews):
                    discounted_sum = r + discounted_sum
                    rtgs.insert(0, discounted_sum)
                batch_weights += rtgs

                obs, _ = env.reset()
                ep_rews = []

                if len(batch_obs) > batch_size:
                    break

        params, opt_state, loss = update_step(
            params, opt_state,
            jnp.array(batch_obs),
            jnp.array(batch_acts),
            jnp.array(batch_weights)
        )
        print(f"Epoch {epoch:2d} | Loss: {loss:.4f} | Avg Return: {np.mean(batch_rets):.2f}")

    env.close()

# Uncomment to run your implementation
train_reinforce()

Epoch  0 | Loss: 7.2676 | Avg Return: 16.50
Epoch  1 | Loss: 21.4653 | Avg Return: 52.26
Epoch  2 | Loss: 21.8216 | Avg Return: 70.14
Epoch  3 | Loss: 29.0367 | Avg Return: 96.37
Epoch  4 | Loss: 33.2270 | Avg Return: 122.60
Epoch  5 | Loss: 40.4581 | Avg Return: 175.76
Epoch  6 | Loss: 39.8089 | Avg Return: 182.07
Epoch  7 | Loss: 41.2542 | Avg Return: 179.43
Epoch  8 | Loss: 45.3734 | Avg Return: 192.56
Epoch  9 | Loss: 55.3385 | Avg Return: 223.96


## Task 3: Vanilla Policy Gradient (VPG) Architecture

REINFORCE works, but it suffers from high variance. If a trajectory is "lucky", we push up the probabilities of all actions in it, even if some of those actions were actually sub-optimal.

We fix this by introducing a **Baseline**. We want to evaluate how an action did *compared to what we expected*. We learn a **Value Function** $V_\phi(s)$ (the Critic), which predicts the expected return from state $s$. We then compute the **Advantage**:

$$A_t = \hat{R}_t - V_\phi(s_t)$$

If $A_t > 0$, the action was better than average! If $A_t < 0$, it was worse.

**Your Task:**
Complete the `ActorCritic` architecture. It should share the same input but branch into two separate networks:
1. An **Actor** network (identical to your PolicyNetwork, returning logits).
2. A **Critic** network (a similar 2-layer MLP, but returning a single scalar value).

In [14]:
class ActorCritic(nn.Module):
    action_dim: int

    @nn.compact
    def __call__(self, x):
        # ==========================================
        # YOUR CODE HERE
        # Define the actor network (returns logits for actions)
        # Define the critic network (returns a single value for the state)
        # ==========================================

        x1 = nn.Dense(64)(x)
        x1 = nn.activation.tanh(x1)
        x1 = nn.Dense(64)(x1)
        x1 = nn.activation.tanh(x1)
        logits = nn.Dense(self.action_dim)(x1)


        x2 = nn.Dense(64)(x)
        x2 = nn.activation.tanh(x2)
        x2 = nn.Dense(64)(x2)
        x2 = nn.activation.tanh(x2)
        value = nn.Dense(1)(x1)
        return logits, value.squeeze()

## Task 4: VPG Loss and Update Step

Now we will implement the update logic for VPG. We need to train the Actor to maximize the expected advantage, and the Critic to accurately predict the rewards-to-go.

**Your Task:**
Inside `update_step`, complete the combined loss function.
1. Calculate the Critic's Mean Squared Error (MSE) loss against `ret_batch`.
2. Calculate the Advantages. **Crucially**, use `jax.lax.stop_gradient` on the values so the actor's update doesn't shift the critic's weights.
3. Compute the Actor's policy loss using the normalized advantages.

In [16]:
def train_vpg(env_id="CartPole-v1", epochs=10, batch_size=5000, lr=1e-2):
    env = gym.make(env_id)
    obs_dim = env.observation_space.shape[0]
    act_dim = env.action_space.n

    ac = ActorCritic(act_dim)
    global key
    key, subkey = jax.random.split(key)

    params = ac.init(subkey, jnp.zeros((1, obs_dim)))
    tx = optax.adam(learning_rate=lr)
    opt_state = tx.init(params)

    @jax.jit
    def update_step(params, opt_state, obs_batch, act_batch, ret_batch):
        def loss_fn(p):
            logits, values = ac.apply(p, obs_batch)

            # ==========================================
            # YOUR CODE HERE
            # 1. Critic Loss (MSE between ret_batch and values)
            # 2. Advantages (ret_batch - stop_gradient(values))
            # 3. Normalize advantages for stability: (A - mean(A)) / (std(A) + 1e-8)
            # 4. Actor Loss (-mean(log_probs * advantages))
            # 5. Return: actor_loss + 0.5 * critic_loss
            # ==========================================
            mse = jnp.mean((values - ret_batch)**2)
            advantages = ret_batch - lx.stop_gradient(values)
            advantages_norm = (advantages - jnp.mean(advantages)) / (jnp.std(advantages)+1e-8)
            log_probs = jax.nn.log_softmax(logits)
            #actor_loss = -jnp.mean(logits * advantages_norm)

            selected_log_probs = log_probs[jnp.arange(batch_size), act_batch]
            actor_loss = -jnp.mean(selected_log_probs * advantages_norm)

            total_loss = mse + 0.5 *actor_loss

            return total_loss, (actor_loss, mse)

        (loss, (pi_loss, v_loss)), grads = jax.value_and_grad(loss_fn, has_aux=True)(params)
        updates, opt_state = tx.update(grads, opt_state, params)
        new_params = optax.apply_updates(params, updates)
        return new_params, opt_state, pi_loss, v_loss

    obs, _ = env.reset(seed=seed)

    for epoch in range(epochs):
        batch_obs, batch_acts, batch_rets, batch_lens = [], [], [], []
        batch_ep_rets = []
        ep_rews = []

        while True:
            batch_obs.append(obs)

            logits, _ = ac.apply(params, jnp.array(obs))
            key, subkey = jax.random.split(key)
            act = jax.random.categorical(subkey, logits).item()

            obs, rew, terminated, truncated, _ = env.step(act)
            batch_acts.append(act)
            ep_rews.append(rew)

            if terminated or truncated:
                batch_lens.append(len(ep_rews))
                batch_ep_rets.append(sum(ep_rews))

                rtgs = []
                discounted_sum = 0
                for r in reversed(ep_rews):
                    discounted_sum = r + discounted_sum
                    rtgs.insert(0, discounted_sum)

                batch_rets += rtgs

                obs, _ = env.reset()
                ep_rews = []

                if len(batch_obs) > batch_size:
                    break

        params, opt_state, pi_loss, v_loss = update_step(
            params, opt_state,
            jnp.array(batch_obs),
            jnp.array(batch_acts),
            jnp.array(batch_rets)
        )

        avg_ret = np.mean(batch_ep_rets) if len(batch_ep_rets) > 0 else 0
        print(f"Epoch {epoch:2d} | Pi Loss: {pi_loss:.4f} | V Loss: {v_loss:.4f} | Avg Return: {avg_ret:.2f}")

    env.close()

train_vpg()

AttributeError: module 'jax.lax' has no attribute 'stop_grad'

## Task 5: Theoretical Understanding

### Question 5a
Run both `train_reinforce()` and `train_vpg()` multiple times or observe their logging output over the 10 epochs. What differences do you notice in the stability of the `Avg Return` across epochs between the two algorithms?

### Question 5b
Why mathematically does introducing the baseline $V_\phi(s)$ in VPG cause the behavior you observed in 5a compared to standard REINFORCE? Describe your reasoning below.

## Task 6: Measuring Gradient Variance (Empirical Proof)

In Task 5, we reasoned that the state-value baseline in VPG centers the advantage, thereby reducing the variance of the policy gradient updates.

Let's prove this empirically. A common way to measure the stability of neural network training is to track the **Global Gradient Norm** (the magnitude of the gradient vectors) at each update step. High variance in the estimator will result in wildly fluctuating gradient sizes.

**Your Task:**
1. Modify your `update_step` functions for both `train_reinforce` and `train_vpg` to calculate the $L_2$ norm of the gradients before they are applied by the optimizer.
2. Return this `grad_norm` alongside the loss.
3. Track these norms over the epochs and plot them using `matplotlib`.

*Hint: JAX stores gradients in a PyTree (a nested dictionary). You can flatten it and compute the norm using:*
```python
leaves = jax.tree_util.tree_leaves(grads)
grad_norm = jnp.sqrt(sum(jnp.sum(x**2) for x in leaves))

In [ ]:
import matplotlib.pyplot as plt

# Assuming you collected `reinforce_grad_norms` and `vpg_grad_norms` lists
# from the modified training loops, you can plot them like this:

def plot_gradient_variance(reinforce_norms, vpg_norms):
    plt.figure(figsize=(10, 5))
    plt.plot(reinforce_norms, label='REINFORCE Gradient Norm', alpha=0.8, color='red')
    plt.plot(vpg_norms, label='VPG Gradient Norm', alpha=0.8, color='blue')

    plt.title('Gradient Variance: REINFORCE vs. VPG')
    plt.xlabel('Epoch')
    plt.ylabel('L2 Norm of Gradients')
    plt.yscale('log') # Log scale helps visualize the massive difference
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

# plot_gradient_variance(reinforce_grad_norms, vpg_grad_norms)